# Repeat Purchase Prediction, part 4
# Model benchmarking and optimisation

## Notebook purpose

This notebook compares more model families with the logistic regression and histogram gradient boosting from `03_modelling_homework.ipynb`, tunes them and picks the model for the final submission. It covers the modelling and evaluation phases of the project.

## Approach

- **Same validation as notebook 3:** four expanding yearly folds (2020–2023) with a 90-day gap, and the holdout from January 2024. Every model is scored on the same rows.
- **Defaults first:** each new model runs with its default settings and is compared with the untuned logistic regression and gradient boosting.
- **Then tuning:** each model is tuned on the development folds and compared with the tuned models of notebook 3.
- **One scoreboard:** F1 at each model's own threshold, the customer-bootstrap interval of the difference, folds where it is worse, ROC-AUC, balanced accuracy and log-loss.
- **Holdout only records:** choices are made on development results; the holdout scores the finalists and changes nothing.

## Structure

1. Setup and data
2. Reference models
3. New models with default settings
4. Tuning
5. Scoreboard and model choice
6. Threshold and calibration
7. Holdout assessment
8. Final fit and submission
9. Summary of decisions

# 1. Setup and data

## 1.1. Paths and experiment settings

In [1]:
# Handle paths without computer-specific locations.
from pathlib import Path

PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir() and (folder / "notebooks").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate the project. Open the notebook within the repository.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "preprocessed"
SUBMISSION_DIR = DATA_DIR / "submissions"
OPTUNA_DIR = DATA_DIR / "optuna"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

for folder in (SUBMISSION_DIR, OPTUNA_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# decision: a new experiment version, so 04 searches never mix with the homework ones
EXPERIMENT_VERSION = "project_v1"

N_TRIALS = 60         # completed trials per search
FRESH_SEARCH = False  # True deletes and restarts the named searches of this version

NEEDED = [
    PROCESSED_DIR / "train_clean.parquet",
    PROCESSED_DIR / "test_clean.parquet",
    PROCESSED_DIR / "decision_log_modelling.json",
    PROCESSED_DIR / "column_roles.json",
    PROCESSED_DIR / "preprocessing_config.json",
    RAW_DIR / "sample_submission.csv",
]

missing = [path.name for path in NEEDED if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}. Run 02_preprocessing and 03_modelling_homework first.")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Experiment version: {EXPERIMENT_VERSION} | trials per search: {N_TRIALS}")

Project root: ml-repeat-purchase
Experiment version: project_v1 | trials per search: 60


## 1.2. Imports, display and plot settings

In [2]:
# Standard library
import sys
import json
import hashlib
import warnings
from time import perf_counter

# Data and numerical calculations
import numpy as np
import pandas as pd

# Charts
import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook display
from IPython.display import display

# Hyperparameter search
import optuna

# Pipelines and custom transformers
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils.validation import check_is_fitted

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    f1_score,
    log_loss,
    roc_auc_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.exceptions import ConvergenceWarning

# Project modules
project_path = str(PROJECT_ROOT)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.repeat_purchase_prep as rpp
from src.decision_log import DecisionLog

RANDOM_STATE = rpp.RANDOM_STATE
target = rpp.TARGET

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# Plot definitions shared by every chart in this notebook
ACCENT = {"burgundy": "#801333", "orange": "#ff8600", "near_black": "#1d212a"}
SERIES_COLOURS = ["#801333", "#f3a481", "#35193e", "#ff8600", "#e13342"]  # one colour per model or group
GRID_COLOUR = "#eeeeee"
FONT = "Inter, Arial, sans-serif"

# Seaborn's rocket palette as Plotly colour scales
ROCKET = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket", 11).as_hex())]
ROCKET_R = [[i / 10, colour] for i, colour in enumerate(sns.color_palette("rocket_r", 11).as_hex())]
# Correlations and loadings: always plotted with zmin=-1, zmax=1
DIVERGING = [[0.0, ACCENT["orange"]], [0.5, "#ffffff"], [1.0, ACCENT["burgundy"]]]

pio.templates["project"] = go.layout.Template(
    layout=dict(
        font=dict(family=FONT, size=13, color=ACCENT["near_black"]),
        title=dict(x=0, xanchor="left", font=dict(size=17)),
        paper_bgcolor="white",
        plot_bgcolor="white",
        colorway=SERIES_COLOURS,
        colorscale=dict(sequential=ROCKET_R, diverging=DIVERGING),
        xaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        yaxis=dict(gridcolor=GRID_COLOUR, zeroline=False, linecolor="#cccccc", ticks="outside"),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font_family=FONT),
        margin=dict(l=60, r=30, t=70, b=60),
    )
)
pio.templates.default = "project"

# decision: figures saved as HTML (plotly.js from the CDN) and PNG at 300 dpi equivalent
HTML_OPTIONS = dict(include_plotlyjs="cdn")
PNG_OPTIONS = dict(scale=300 / 72)

## 1.3. Data and folds

The cleaned data, column roles and validation settings come from notebook 2. The folds are the four expanding years used in notebook 3, so every model here is scored on the same rows. Only the key checks are repeated.

In [4]:
train = pd.read_parquet(PROCESSED_DIR / "train_clean.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test_clean.parquet")
sample_submission = pd.read_csv(RAW_DIR / "sample_submission.csv")

config = json.loads((PROCESSED_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
roles = json.loads((PROCESSED_DIR / "column_roles.json").read_text(encoding="utf-8"))

# Validation settings saved by notebook 2
GAP_DAYS = int(config["validation"]["gap_days"])
HOLDOUT_START = pd.Timestamp(config["validation"]["holdout_start"])
DEV_END = HOLDOUT_START - pd.Timedelta(days=GAP_DAYS)
VAL_YEARS = list(config["validation"]["validation_years"])

dev = train.loc[train["purchase_date"] < DEV_END].reset_index(drop=True)
holdout = train.loc[train["purchase_date"] >= HOLDOUT_START].reset_index(drop=True)

# Expanding yearly folds: training outcomes close before the validation year starts
folds = []
for year in VAL_YEARS:
    start = pd.Timestamp(year=year, month=1, day=1)
    stop = min(pd.Timestamp(year=year + 1, month=1, day=1), DEV_END)
    train_idx = np.flatnonzero(dev["purchase_date"] < start - pd.Timedelta(days=GAP_DAYS))
    val_idx = np.flatnonzero((dev["purchase_date"] >= start) & (dev["purchase_date"] < stop))
    folds.append((train_idx, val_idx))

# Key checks only; the full set ran in notebook 3
assert set(train.columns) - {target} == set(test.columns)
assert set(sample_submission["ID"]) == set(test["ID"])
assert roles == rpp.column_roles(dev), "Saved column roles no longer match the data."

log = DecisionLog.load(PROCESSED_DIR / "decision_log_modelling.json")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Development: {len(dev):,} | Holdout: {len(holdout):,}")
print("Fold sizes (train / validation):", {y: (len(tr), len(va)) for y, (tr, va) in zip(VAL_YEARS, folds)})
print(f"Decisions loaded: {len(log)}")

Train: (6469, 53) | Test: (986, 52)
Development: 5,403 | Holdout: 835
Fold sizes (train / validation): {2020: (2578, 670), 2021: (3234, 782), 2022: (4004, 628), 2023: (4705, 536)}
Decisions loaded: 38


## 1.4. Pipeline builders

Two input branches, as decided in notebook 2. The linear branch fills, scales and encodes the columns, and can add the elastic-net selection. The tree branch keeps missing values and codes the categories. Distance and probability models (KNN, Naive Bayes, LDA) use the linear branch; trees use the tree branch.

In [5]:
# Saved branch decisions from notebook 2
NOISE = list(config["excluded_noise"])
TREE_DROP = list(config["hist_gradient_boosting"]["drop_columns"])


def branch_roles(branch):
    """Numeric, large and small categorical inputs of one branch."""
    excluded = set(NOISE + (TREE_DROP if branch == "tree" else []))
    return {group: [c for c in roles[group] if c not in excluded] for group in ("num", "high", "low")}


def branch_inputs(branch):
    r = branch_roles(branch)
    return r["num"] + r["high"] + r["low"]


def build_linear_pipeline(model, select=True):
    """Median fill and scaling, frequency and one-hot encoding, optional elastic-net selection."""
    settings, r = config["linear"], branch_roles("linear")
    prep = ColumnTransformer(
        [
            ("num", Pipeline([
                ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scale", StandardScaler()),
            ]), r["num"]),
            ("high", Pipeline([("encode", rpp.FrequencyEncoder()), ("scale", StandardScaler())]), r["high"]),
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=settings["rare_min_rows"]), r["low"]),
        ],
        sparse_threshold=0,
    )
    steps = [("prep", prep)]
    if select:
        steps.append(("select", SelectFromModel(
            LogisticRegression(solver="saga", l1_ratio=settings["selector_l1_ratio"], C=settings["selector_C"],
                               max_iter=5000, random_state=RANDOM_STATE),
            threshold=settings["selector_threshold"],
        )))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def build_tree_pipeline(model):
    """Small categories as codes (first), large ones as frequencies, numbers untouched; missing values kept."""
    r = branch_roles("tree")
    prep = ColumnTransformer(
        [
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), r["low"]),
            ("high", rpp.FrequencyEncoder(), r["high"]),
            ("num", "passthrough", r["num"]),
        ],
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )
    model = clone(model)
    if isinstance(model, HistGradientBoostingClassifier):
        # decision: native categories and no internal random split, as in notebook 3
        model.set_params(categorical_features=list(range(len(r["low"]))), early_stopping=False)
    return Pipeline([("prep", prep), ("model", model)])


print("Linear inputs:", len(branch_inputs("linear")), "| Tree inputs:", len(branch_inputs("tree")))

Linear inputs: 41 | Tree inputs: 38


## 1.5. Scoring on the folds

Each model is fitted in every fold and scored on the later year. Its threshold is chosen once, on the four years' predictions pooled, with the same rule as notebook 3. The predictions keep the customer and whether that customer appeared in the training part, for the error analysis.

In [6]:
def best_threshold(y, p, tol=0.005):
    """Centre of the longest run of thresholds within tol of the best F1 (grid 0.05 to 0.95)."""
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    actual = np.asarray(y).astype(bool)[:, None]
    predicted = np.asarray(p)[:, None] >= grid
    tp = (predicted & actual).sum(axis=0)
    fp = (predicted & ~actual).sum(axis=0)
    fn = (~predicted & actual).sum(axis=0)
    f1s = 2 * tp / np.maximum(2 * tp + fp + fn, 1)

    near = np.flatnonzero(f1s >= f1s.max() - tol)
    runs = [r for r in np.split(near, np.flatnonzero(np.diff(near) > 1) + 1) if np.any(f1s[r] == f1s.max())]
    chosen = max(runs, key=lambda r: (len(r), -int(r[0])))
    return float(grid[chosen[(len(chosen) - 1) // 2]]), grid, f1s


def f1_floor(y):
    """F1 of predicting a repeat for every row."""
    share = np.mean(y)
    return 2 * share / (1 + share)


def score_folds(pipe, inputs, data=dev):
    """Fit in every fold; return (fold table, out-of-fold predictions, pooled threshold)."""
    rows, oof = [], []
    for (tr, va), year in zip(folds, VAL_YEARS):
        train_part, val_part = data.iloc[tr], data.iloc[va]
        start = perf_counter()
        model = clone(pipe).fit(train_part[inputs], train_part[target])
        seconds = perf_counter() - start

        p = model.predict_proba(val_part[inputs])[:, 1]
        p_train = model.predict_proba(train_part[inputs])[:, 1]
        y = val_part[target].to_numpy()
        steps = model.named_steps
        rows.append({
            "fold": year,
            "n_features": (int(steps["select"].get_support().sum()) if "select" in steps
                           else len(steps["prep"].get_feature_names_out())),
            "log_loss": log_loss(y, p, labels=[0, 1]),
            "train_log_loss": log_loss(train_part[target], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(y, p),
            "fit_seconds": seconds,
        })
        # Kept for later error analysis: who the customer is and whether training saw them
        oof.append(pd.DataFrame({
            "fold": year, "ID": val_part["ID"].to_numpy(), "customer_id": val_part["customer_id"].to_numpy(),
            "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(), "y": y, "p": p,
        }))

    oof = pd.concat(oof, ignore_index=True)
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold


def summary_row(table, oof, threshold):
    """One line per model: pooled F1 at its own threshold plus fold averages."""
    predicted = oof["p"] >= threshold
    return {
        "n_features": table["n_features"].mean(),
        "f1": f1_score(oof["y"], predicted),
        "floor": f1_floor(oof["y"]),
        "balanced_accuracy": balanced_accuracy_score(oof["y"], predicted),
        "roc_auc": table["roc_auc"].mean(),
        "log_loss": table["log_loss"].mean(),
        "overfit_gap": (table["log_loss"] - table["train_log_loss"]).mean(),
        "threshold": threshold,
        "fit_seconds": table["fit_seconds"].sum(),
    }

## 1.6. Decision rule

Same rule as notebook 3: an option beats the reference only if the 95% customer-bootstrap interval of the F1 difference sits above zero and it is worse in at most one fold.

In [7]:
def fast_f1(y, predicted):
    tp = np.sum(y & predicted)
    return 2 * tp / (2 * tp + np.sum(~y & predicted) + np.sum(y & ~predicted))


def bootstrap_f1_diff(oof_ref, oof_opt, thr_ref, thr_opt, n_boot=2000, seed=RANDOM_STATE):
    """95% interval of F1(option) - F1(reference), resampling whole customers."""
    assert oof_ref["ID"].equals(oof_opt["ID"]), "Options must be scored on the same rows."
    y = oof_ref["y"].to_numpy(dtype=bool)
    pred_ref, pred_opt = oof_ref["p"].to_numpy() >= thr_ref, oof_opt["p"].to_numpy() >= thr_opt
    groups = list(oof_ref.groupby("customer_id", sort=False).indices.values())

    rng = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        diffs[b] = fast_f1(y[rows], pred_opt[rows]) - fast_f1(y[rows], pred_ref[rows])
    return np.percentile(diffs, [2.5, 97.5])


def judge(runs, reference):
    """Scoreboard against the reference: better only if the interval is above zero and at most one fold is worse."""
    ref_table, ref_oof, ref_thr = runs[reference]
    rows = []
    for name, (table, oof, thr) in runs.items():
        row = {"model": name, **summary_row(table, oof, thr)}
        row["folds_worse"] = int((table["f1"] < ref_table["f1"]).sum())
        if name == reference:
            row.update(ci_low=np.nan, ci_high=np.nan, verdict="reference")
        else:
            lo, hi = bootstrap_f1_diff(ref_oof, oof, ref_thr, thr)
            verdict = ("better" if lo > 0 and row["folds_worse"] <= 1
                       else "worse" if hi < 0 else "no clear difference")
            row.update(ci_low=lo, ci_high=hi, verdict=verdict)
        rows.append(row)

    board = pd.DataFrame(rows).set_index("model")
    board.insert(2, "f1_diff", board["f1"] - board.loc[reference, "f1"])
    return board

# 2. Reference models

Three models set the bar for everything that follows. The dummy always predicts the training share of repeats, so its F1 is the "always repeat" floor. The untuned logistic regression and gradient boosting are the starting points of notebook 3. Logistic regression is the reference: a new model has to beat it under the decision rule.

## 2.1. Scoreboard

In [8]:
linear_inputs, tree_inputs = branch_inputs("linear"), branch_inputs("tree")

reference_pipes = {
    # decision: the dummy predicts the training share, so its F1 is the always-repeat floor
    "dummy": (build_linear_pipeline(DummyClassifier(strategy="prior"), select=False), linear_inputs),
    "logistic": (build_linear_pipeline(LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)), linear_inputs),
    "hist_gb": (build_tree_pipeline(HistGradientBoostingClassifier(random_state=RANDOM_STATE)), tree_inputs),
}

reference_runs = {name: score_folds(pipe, inputs) for name, (pipe, inputs) in reference_pipes.items()}

board_reference = judge(reference_runs, reference="logistic")
board_reference.loc["dummy", "n_features"] = 0  # it ignores every column
display(board_reference)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
dummy,0.000,0.709,-0.042,0.709,0.500,0.500,0.689,0.002,0.300,0.264,4,-0.059,-0.021,worse
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
hist_gb,38.000,0.736,-0.015,0.709,0.654,0.760,0.607,0.348,0.300,2.321,4,-0.031,0.001,no clear difference


Logistic regression reaches F1 0.750 against the floor of 0.709, with ROC-AUC 0.784 and balanced accuracy 0.684. Gradient boosting scores 0.736: its interval [-0.031, 0.001] includes zero, so the difference is not clear, but it is worse in all four years. The dummy is clearly worse, as expected, with balanced accuracy 0.500 and ROC-AUC 0.500.

## 2.2. F1 by validation year

In [9]:
fold_f1 = pd.DataFrame({name: reference_runs[name][0]["f1"] for name in ["logistic", "hist_gb"]})
fold_f1["always repeat"] = [f1_floor(part["y"]) for _, part in reference_runs["logistic"][1].groupby("fold")]

fig = go.Figure()
for name in ["logistic", "hist_gb", "always repeat"]:
    fig.add_trace(go.Scatter(
        x=fold_f1.index, y=fold_f1[name], mode="lines+markers" if name != "always repeat" else "lines",
        name=name, line=dict(dash="dash", color="grey") if name == "always repeat" else None,
        hovertemplate="%{x}: F1 %{y:.3f}<extra>" + name + "</extra>",
    ))
fig.update_layout(
    title="Reference models: F1 in each validation year",
    xaxis=dict(title="Validation year", tickmode="array", tickvals=VAL_YEARS),
    yaxis=dict(title="F1 at each model's pooled threshold"),
    height=420,
)
fig.write_html(FIGURES_DIR / "04_reference_f1_by_year.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_reference_f1_by_year.png", **PNG_OPTIONS)
fig.show()

In [10]:
display(fold_f1.round(3))

,logistic,hist_gb,always repeat
fold,,,
2020,0.758,0.742,0.745
2021,0.764,0.764,0.709
2022,0.716,0.689,0.649
2023,0.760,0.740,0.728


Logistic regression stays above the floor in every year, from 0.716 in 2022 to 0.764 in 2021. Its smallest lift is in 2020 (0.758 against 0.745). Gradient boosting ties it in 2021 (0.764) and falls to 0.689 in 2022, the year with the lowest floor (0.649).

## Findings

| Model | F1 | ROC-AUC | Balanced accuracy | Verdict against logistic |
|---|---|---|---|---|
| Dummy (always repeat) | 0.709 | 0.500 | 0.500 | worse |
| Logistic regression | 0.750 | 0.784 | 0.684 | reference |
| Gradient boosting | 0.736 | 0.760 | 0.654 | no clear difference, worse in 4 of 4 years |

- The untuned logistic regression is the reference for the default models in section 3, at F1 0.750.
- The floor is high (0.709), because about 56% of development purchases are followed by a repeat. Balanced accuracy and ROC-AUC show the gap to a model without information more clearly than F1.
- Gradient boosting's log-loss gap between training and validation (0.348, against 0.009 for logistic regression) shows its default settings follow the training years closely.

# 3. New models with default settings

Each model runs with its library defaults on the same folds, so the comparison shows what each model family brings before any tuning. A short check before each model tests the assumption it relies on.

## 3.1. Naive Bayes

Gaussian Naive Bayes assumes that, within each class, every column is normal and independent of the others. The check measures how far the prepared columns are from that.

In [11]:
# Prepared linear-branch columns of the last fold, the same matrix Naive Bayes sees
tr, _ = folds[-1]
nb_prep = build_linear_pipeline(GaussianNB(), select=False).named_steps["prep"]
X_nb = pd.DataFrame(
    nb_prep.fit_transform(dev.iloc[tr][linear_inputs]),
    columns=nb_prep.get_feature_names_out(),
)
y_nb = dev.iloc[tr][target].to_numpy()

binary = [c for c in X_nb if set(np.unique(X_nb[c])) <= {0.0, 1.0}]
numeric = [c for c in X_nb if c not in binary]
skew_by_class = pd.DataFrame({
    "skew_no_repeat": X_nb.loc[y_nb == 0, numeric].skew(),
    "skew_repeat": X_nb.loc[y_nb == 1, numeric].skew(),
})
skew_by_class["max_abs_skew"] = skew_by_class.abs().max(axis=1)

# Strongest correlation of each numeric column with another, inside the repeat class
corr = X_nb.loc[y_nb == 1, numeric].corr().abs().where(lambda m: ~np.eye(len(m), dtype=bool))
skew_by_class["max_abs_corr"] = corr.max()

print(f"{len(X_nb.columns)} prepared columns: {len(binary)} take only 0 and 1, {len(numeric)} are numeric")
print(f"Numeric columns with |skew| > 1 in either class: {(skew_by_class['max_abs_skew'] > 1).sum()}")
print(f"Numeric columns correlated above 0.8 with another: {(skew_by_class['max_abs_corr'] > 0.8).sum()}")
display(skew_by_class.sort_values("max_abs_skew", ascending=False).head(8).round(2))

73 prepared columns: 39 take only 0 and 1, 34 are numeric
Numeric columns with |skew| > 1 in either class: 30
Numeric columns correlated above 0.8 with another: 17


,skew_no_repeat,skew_repeat,max_abs_skew,max_abs_corr
num__Total Desc_ Global,31.360,24.280,31.360,0.130
num__Total Portes,15.700,24.020,24.020,0.270
num__prior_std_gap_days,5.080,9.020,9.020,0.640
num__Total Desc_ Linha,7.840,6.460,7.840,0.630
num__invoice_count_current_purchase,7.610,5.690,7.610,0.170
num__spend_30d,7.100,3.820,7.100,0.710
num__prior_mean_gap_days,4.530,6.970,6.970,0.640
num__prior_purchase_count,6.330,2.980,6.330,0.870


Both assumptions fail. Of the 73 prepared columns, 39 take only the values 0 and 1, so they cannot be normal. Of the 34 numeric columns, 30 have a skew above 1 in at least one class (up to 31.4 for `Total Desc_ Global`), and 17 correlate above 0.8 with another column, so the same information is counted several times.

In [12]:
nb_runs = {
    "naive_bayes": score_folds(build_linear_pipeline(GaussianNB(), select=False), linear_inputs),
    "naive_bayes_selected": score_folds(build_linear_pipeline(GaussianNB()), linear_inputs),
}
board_nb = judge({"logistic": reference_runs["logistic"], **nb_runs}, reference="logistic")
display(board_nb)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
naive_bayes,69.250,0.673,-0.078,0.709,0.650,0.725,6.543,2.308,0.120,0.207,4,-0.124,-0.047,worse
naive_bayes_selected,20.000,0.703,-0.048,0.709,0.703,0.756,2.568,0.447,0.060,3.609,4,-0.098,-0.019,worse


On all 69 prepared columns Naive Bayes scores F1 0.673, below the always-repeat floor of 0.709. With the 20 elastic-net columns it reaches 0.703, still below the floor. Both are clearly worse than logistic regression. The probabilities are badly overconfident: log-loss is 6.54 and 2.57 against 0.561, and the selected thresholds sit at 0.12 and 0.06, near the bottom of the grid.

## 3.2. Linear discriminant analysis

LDA fits one normal distribution per class with a shared covariance matrix, and it has to invert that matrix. Strongly or exactly collinear columns make the inversion unstable, so the check measures collinearity.

In [13]:
# Collinearity of the same prepared matrix (last fold), all columns together
X_all = X_nb.to_numpy()
X_all = X_all[:, X_all.std(axis=0) > 0]  # constant columns carry no information
singular_values = np.linalg.svd(X_all - X_all.mean(axis=0), compute_uv=False)
condition_number = singular_values[0] / singular_values[-1]

# Pairs of numeric columns that move almost together
pairs = (
    X_nb[numeric].corr().abs()
    .where(np.triu(np.ones((len(numeric), len(numeric)), dtype=bool), k=1))
    .stack().sort_values(ascending=False)
)

print(f"Condition number of the centred matrix: {condition_number:,.0f}")
print(f"Numeric pairs with |correlation| > 0.9: {(pairs > 0.9).sum()}")
display(pairs.head(6).round(3).rename("abs_correlation").to_frame())

Condition number of the centred matrix: 91,724,876,045,469,680
Numeric pairs with |correlation| > 0.9: 5


,,abs_correlation
num__Total a Pagar,num__Total Liquido,0.996
num__Total Bruto,num__Total Liquido,0.982
num__Total a Pagar,num__Total Bruto,0.972
num__purchase_count_180d,num__purchase_count_365d,0.954
num__purchase_count_90d,num__purchase_count_180d,0.936
num__Total a Pagar,num__Total Imp_,0.894


The condition number is about 4 × 10¹⁷, which means the matrix is singular: some columns are exact combinations of others. Two causes: the one-hot encoding keeps every level, so each group of indicators sums to 1, and the accounting identity from the EDA makes `Total a Pagar` equal to `Total Liquido` plus `Total Imp_` (correlation 0.996). Five numeric pairs correlate above 0.9, including the 90-, 180- and 365-day purchase counts.

In [14]:
lda_runs = {
    "lda": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False), linear_inputs),
    "lda_selected": score_folds(build_linear_pipeline(LinearDiscriminantAnalysis()), linear_inputs),
}
board_lda = judge({"logistic": reference_runs["logistic"], **lda_runs}, reference="logistic")
display(board_lda)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
lda,69.250,0.733,-0.018,0.709,0.645,0.741,0.616,0.052,0.350,0.310,3,-0.031,-0.006,worse
lda_selected,20.000,0.733,-0.017,0.709,0.613,0.752,0.593,0.021,0.290,3.380,4,-0.030,-0.002,worse


LDA scores F1 0.733 on all columns and 0.733 on the 20 selected ones, above the floor of 0.709 but clearly below logistic regression (intervals [-0.031, -0.006] and [-0.030, -0.002]). Its probabilities are far better behaved than Naive Bayes's: log-loss 0.616 and 0.593, with thresholds of 0.35 and 0.29. The default `svd` solver runs despite the singular matrix, because it never inverts the covariance directly.

## 3.3. K-nearest neighbours

KNN predicts from the training rows closest to each new row, so it relies on distances being meaningful. With many columns, the nearest and farthest neighbours can end up almost equally far away. The check compares the two distances on all prepared columns and on the elastic-net selection.

In [15]:
# Distance concentration on the last fold: as columns grow, nearest and farthest neighbours look alike
tr, va = folds[-1]
selected_pipe = build_linear_pipeline(KNeighborsClassifier()).fit(dev.iloc[tr][linear_inputs], dev.iloc[tr][target])

matrices = {
    "all columns": (X_nb.to_numpy(), nb_prep.transform(dev.iloc[va][linear_inputs])),
    "selected columns": tuple(
        selected_pipe.named_steps["select"].transform(selected_pipe.named_steps["prep"].transform(dev.iloc[part][linear_inputs]))
        for part in (tr, va)
    ),
}

rows = []
for name, (X_train_fold, X_val_fold) in matrices.items():
    distances = np.linalg.norm(X_val_fold[:, None, :] - X_train_fold[None, :, :], axis=2)
    nearest, farthest = distances.min(axis=1), distances.max(axis=1)
    rows.append({
        "columns": X_train_fold.shape[1],
        "median_nearest": np.median(nearest),
        "median_farthest": np.median(farthest),
        "median_contrast": np.median((farthest - nearest) / nearest),
    })
display(pd.DataFrame(rows, index=list(matrices)).round(2))

,columns,median_nearest,median_farthest,median_contrast
all columns,73,2.230,48.690,20.770
selected columns,22,1.120,17.230,14.500


The contrast between the farthest and nearest neighbour is higher with all 73 columns (20.8) than with the 22 selected ones (14.5), the opposite of the expected concentration. The farthest distances are driven by the extreme values seen in the Naive Bayes check (skew up to 31), so this measure mostly reflects outliers rather than the number of columns. The selection halves the distance to the nearest neighbour (median 1.12 against 2.23).

In [16]:
knn_runs = {
    "knn": score_folds(build_linear_pipeline(KNeighborsClassifier(), select=False), linear_inputs),
    "knn_selected": score_folds(build_linear_pipeline(KNeighborsClassifier()), linear_inputs),
}
board_knn = judge({"logistic": reference_runs["logistic"], **knn_runs}, reference="logistic")
display(board_knn)

print("Distinct probabilities from the selected model:", np.sort(knn_runs["knn_selected"][1]["p"].unique()))

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
knn,69.250,0.726,-0.024,0.709,0.632,0.734,1.981,1.557,0.300,0.287,4,-0.041,-0.008,worse
knn_selected,20.000,0.732,-0.018,0.709,0.643,0.731,2.130,1.709,0.300,3.460,3,-0.036,-0.001,worse


Distinct probabilities from the selected model: [0.  0.2 0.4 0.6 0.8 1. ]


With five neighbours, KNN scores F1 0.726 on all columns and 0.732 on the selected ones, both above the floor of 0.709 and clearly below logistic regression. It gives only six probabilities (0, 0.2, … 1), so its threshold of 0.30 means "at least two of the five neighbours repeated". Many predictions at exactly 0 or 1 explain the high log-loss (about 2.0), and the training-validation gap of 1.6–1.7 shows each training row votes for itself.

## 3.4. Decision tree

A tree with default settings keeps splitting until every leaf is pure, so it can memorise the training rows. The check measures how large the default tree grows in each fold and how many leaves hold a single row. Missing values stay in: the tree sends them down whichever branch fits the training rows better.

In [17]:
# Size of the default tree in each fold, fitted on the tree branch with missing values kept
tree_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    fitted = build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)).fit(
        dev.iloc[tr][tree_inputs], dev.iloc[tr][target]
    )
    tree = fitted.named_steps["model"]
    leaf_sizes = tree.tree_.n_node_samples[tree.tree_.children_left == -1]
    tree_rows.append({
        "fold": year,
        "training_rows": len(tr),
        "depth": tree.get_depth(),
        "leaves": tree.get_n_leaves(),
        "median_leaf_size": np.median(leaf_sizes),
        "single_row_leaves": np.mean(leaf_sizes == 1),
    })
display(pd.DataFrame(tree_rows).set_index("fold").round(3))

,training_rows,depth,leaves,median_leaf_size,single_row_leaves
fold,,,,,
2020,2578,24,463,3.000,0.348
2021,3234,26,583,2.000,0.341
2022,4004,26,693,2.000,0.375
2023,4705,33,830,2.000,0.402


The default tree grows with the data: from depth 24 and 463 leaves in the 2020 fold to depth 33 and 830 leaves in 2023. The median leaf holds two or three training rows, and 34–40% of the leaves hold a single row, so most answers rest on one or two purchases.

In [18]:
tree_runs = {"decision_tree": score_folds(build_tree_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)), tree_inputs)}
board_tree = judge({"logistic": reference_runs["logistic"], **tree_runs}, reference="logistic")
display(board_tree)

print("Distinct probabilities:", tree_runs["decision_tree"][1]["p"].nunique())

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
decision_tree,38.000,0.626,-0.124,0.709,0.594,0.592,14.692,14.692,0.500,0.292,4,-0.161,-0.092,worse


Distinct probabilities: 2


The default tree scores F1 0.626, well below the floor of 0.709, with ROC-AUC 0.592 and balanced accuracy 0.594. Its pure leaves give only two probabilities, 0 and 1, so the threshold has no effect and log-loss is extreme (14.7). The training-validation gap equals the validation log-loss, because the tree fits its training rows perfectly.

## 3.5. Random forest

A random forest averages many deep trees, each grown on a bootstrap sample of the rows. The rows left out of each sample give an out-of-bag score at no extra cost, but those rows are drawn at random from the same years. The check compares that score with F1 on the later validation year, to see whether random-row scoring would flatter the model here.

In [19]:
# Out-of-bag F1 (random rows of the training years) against F1 on the later validation year
oob_rows = []
for (tr, va), year in zip(folds, VAL_YEARS):
    forest_pipe = build_tree_pipeline(RandomForestClassifier(oob_score=True, random_state=RANDOM_STATE, n_jobs=-1))
    forest_pipe.fit(dev.iloc[tr][tree_inputs], dev.iloc[tr][target])
    forest = forest_pipe.named_steps["model"]

    p_oob = forest.oob_decision_function_[:, 1]
    p_val = forest_pipe.predict_proba(dev.iloc[va][tree_inputs])[:, 1]
    oob_rows.append({
        "fold": year,
        "oob_f1": f1_score(dev.iloc[tr][target], p_oob >= 0.5),
        "validation_f1": f1_score(dev.iloc[va][target], p_val >= 0.5),
        "median_tree_depth": np.median([t.get_depth() for t in forest.estimators_]),
    })
oob_table = pd.DataFrame(oob_rows).set_index("fold")
oob_table["optimism"] = oob_table["oob_f1"] - oob_table["validation_f1"]
display(oob_table.round(3))

,oob_f1,validation_f1,median_tree_depth,optimism
fold,,,,
2020,0.723,0.722,25.000,0.001
2021,0.722,0.741,26.000,-0.018
2022,0.734,0.724,27.000,0.011
2023,0.728,0.703,28.000,0.025


At a 0.5 cut-off, the out-of-bag F1 differs from the later year's F1 by -0.018 to +0.025, with no consistent direction. On this data, random-row scoring does not clearly flatter the forest, but the chronological folds remain the basis for every decision. The trees grow to a median depth of 25–28, as deep as the single default tree.

In [20]:
forest_runs = {
    "random_forest": score_folds(
        build_tree_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)), tree_inputs
    )
}
board_forest = judge({"logistic": reference_runs["logistic"], **forest_runs}, reference="logistic")
display(board_forest)

,n_features,f1,f1_diff,floor,balanced_accuracy,roc_auc,log_loss,overfit_gap,threshold,fit_seconds,folds_worse,ci_low,ci_high,verdict
model,,,,,,,,,,,,,,
logistic,20.000,0.750,0.000,0.709,0.684,0.784,0.561,0.009,0.340,3.439,0,NaN,NaN,reference
random_forest,38.000,0.736,-0.015,0.709,0.669,0.760,0.589,0.433,0.440,0.793,4,-0.029,-0.002,worse


The default forest scores F1 0.736, the same as gradient boosting and far above the single tree's 0.626: averaging 100 deep trees removes most of their individual noise. It is still clearly worse than logistic regression (interval [-0.029, -0.002]) and below it in all four years. Its probabilities are usable (log-loss 0.589, 94 distinct values), but its training-validation gap of 0.433 shows the deep trees still follow the training years closely.

## 3.6. Support vector machine (pending)

## 3.7. Multilayer perceptron (pending)

## 3.8. Default scoreboard

In [21]:
default_runs = {**reference_runs, **nb_runs, **lda_runs, **knn_runs, **tree_runs, **forest_runs}
board_default = judge(default_runs, reference="logistic").sort_values("f1", ascending=False)
board_default.loc["dummy", "n_features"] = 0
display(board_default[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse",
                       "balanced_accuracy", "roc_auc", "log_loss", "threshold", "verdict"]])

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,balanced_accuracy,roc_auc,log_loss,threshold,verdict
model,,,,,,,,,,,
logistic,20.000,0.750,0.000,NaN,NaN,0,0.684,0.784,0.561,0.340,reference
random_forest,38.000,0.736,-0.015,-0.029,-0.002,4,0.669,0.760,0.589,0.440,worse
hist_gb,38.000,0.736,-0.015,-0.031,0.001,4,0.654,0.760,0.607,0.300,no clear difference
lda_selected,20.000,0.733,-0.017,-0.030,-0.002,4,0.613,0.752,0.593,0.290,worse
lda,69.250,0.733,-0.018,-0.031,-0.006,3,0.645,0.741,0.616,0.350,worse
knn_selected,20.000,0.732,-0.018,-0.036,-0.001,3,0.643,0.731,2.130,0.300,worse
knn,69.250,0.726,-0.024,-0.041,-0.008,4,0.632,0.734,1.981,0.300,worse
dummy,0.000,0.709,-0.042,-0.059,-0.021,4,0.500,0.500,0.689,0.300,worse
naive_bayes_selected,20.000,0.703,-0.048,-0.098,-0.019,4,0.703,0.756,2.568,0.060,worse


In [22]:
order = board_default.index[::-1]  # best at the top of the chart
fig = go.Figure(go.Scatter(
    x=board_default.loc[order, "f1_diff"], y=order, mode="markers",
    marker=dict(size=10, color=ACCENT["burgundy"]),
    error_x=dict(
        type="data", symmetric=False,
        array=(board_default.loc[order, "ci_high"] - board_default.loc[order, "f1_diff"]).fillna(0),
        arrayminus=(board_default.loc[order, "f1_diff"] - board_default.loc[order, "ci_low"]).fillna(0),
        color=ACCENT["near_black"], thickness=1.2,
    ),
    customdata=board_default.loc[order, ["f1", "ci_low", "ci_high"]],
    hovertemplate="%{y}<br>F1 %{customdata[0]:.3f}<br>difference %{x:+.3f}"
                  "<br>95% interval [%{customdata[1]:.3f}, %{customdata[2]:.3f}]<extra></extra>",
))
fig.add_vline(x=0, line=dict(color="grey", dash="dash"))
fig.update_layout(
    title="Default models: F1 difference from logistic regression, 95% customer-bootstrap interval",
    xaxis_title="F1 minus logistic regression's F1",
    height=460,
)
fig.write_html(FIGURES_DIR / "04_default_scoreboard.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_default_scoreboard.png", **PNG_OPTIONS)
fig.show()

No default model beats logistic regression. Every interval lies below zero or touches it, and gradient boosting is the only "no clear difference" ([-0.031, 0.001]). Random forest, gradient boosting, LDA and KNN cluster at F1 0.726–0.736. Naive Bayes and the single tree fall below the always-repeat floor.

## Findings

| Model | F1 | Difference [95% interval] | ROC-AUC | Main issue at defaults |
|---|---|---|---|---|
| Logistic regression | 0.750 | reference | 0.784 | — |
| Random forest | 0.736 | -0.015 [-0.029, -0.002] | 0.760 | deep trees, gap 0.433 |
| Gradient boosting | 0.736 | -0.015 [-0.031, 0.001] | 0.760 | gap 0.348 |
| LDA (selected) | 0.733 | -0.017 [-0.030, -0.002] | 0.752 | singular covariance |
| KNN (selected) | 0.732 | -0.018 [-0.036, -0.001] | 0.731 | six probability values |
| Naive Bayes (selected) | 0.703 | -0.048 [-0.098, -0.019] | 0.756 | overconfident probabilities |
| Decision tree | 0.626 | -0.124 [-0.161, -0.092] | 0.592 | one-row leaves, 0/1 probabilities |

- At default settings, logistic regression stays the best model. Each other family fails in a way its check predicted: overconfidence for Naive Bayes, collinearity for LDA, coarse votes for KNN and overfitting for the trees.
- The elastic-net selection helped Naive Bayes and KNN and did not change LDA, so the selected versions go forward.
- Every failure has a matching setting to tune: calibration and smoothing, shrinkage, the number of neighbours, and pruning or depth limits.

# 4. Tuning

Each model family is tuned on the development folds with two Optuna searches of 60 trials each: one steered by log-loss and one by F1. Every trial records both. The preprocessing is fitted once per fold and reused by every trial, because only the model's settings change. The five best distinct trials of each search are rescored and compared with the family's default model under the decision rule.

## 4.0. Search setup

In [23]:
# Fingerprint of the saved inputs: a change in data or configuration starts new searches
fingerprint = hashlib.sha256(EXPERIMENT_VERSION.encode("utf-8"))
for filename in ["train_clean.parquet", "test_clean.parquet", "column_roles.json", "preprocessing_config.json"]:
    fingerprint.update((PROCESSED_DIR / filename).read_bytes())
INPUT_FINGERPRINT = fingerprint.hexdigest()

STORAGE = f"sqlite:///{(OPTUNA_DIR / f'studies_{EXPERIMENT_VERSION}_{INPUT_FINGERPRINT[:12]}.db').as_posix()}"
STUDY_METADATA = {
    "experiment_version": EXPERIMENT_VERSION,
    "input_fingerprint": INPUT_FINGERPRINT,
    "validation_years": [int(year) for year in VAL_YEARS],
    "gap_days": GAP_DAYS,
}
OBJECTIVES = {"log_loss": "minimize", "f1": "maximize"}


class FoldCache:
    """Preprocessing fitted once per fold on the training rows; matrices reused by every trial."""

    def __init__(self, prep, inputs, data=dev):
        self.folds = []
        for (tr, va), year in zip(folds, VAL_YEARS):
            train_part, val_part = data.iloc[tr], data.iloc[va]
            fitted = clone(prep).fit(train_part[inputs], train_part[target])
            self.folds.append({
                "year": year,
                "X_train": np.asarray(fitted.transform(train_part[inputs]), dtype=float),
                "y_train": train_part[target].to_numpy(),
                "X_val": np.asarray(fitted.transform(val_part[inputs]), dtype=float),
                "y_val": val_part[target].to_numpy(),
                "ID": val_part["ID"].to_numpy(),
                "customer_id": val_part["customer_id"].to_numpy(),
                "seen": val_part["customer_id"].isin(train_part["customer_id"]).to_numpy(),
            })


def evaluate_cached(cache, model):
    """Same output as score_folds, on cached matrices."""
    rows, oof = [], []
    for fold in cache.folds:
        start = perf_counter()
        fitted = clone(model).fit(fold["X_train"], fold["y_train"])
        seconds = perf_counter() - start
        p = fitted.predict_proba(fold["X_val"])[:, 1]
        p_train = fitted.predict_proba(fold["X_train"])[:, 1]
        rows.append({
            "fold": fold["year"],
            "n_features": fold["X_train"].shape[1],
            "log_loss": log_loss(fold["y_val"], p, labels=[0, 1]),
            "train_log_loss": log_loss(fold["y_train"], p_train, labels=[0, 1]),
            "roc_auc": roc_auc_score(fold["y_val"], p),
            "fit_seconds": seconds,
        })
        oof.append(pd.DataFrame({key: fold[key] for key in ["ID", "customer_id", "seen"]}
                                | {"fold": fold["year"], "y": fold["y_val"], "p": p}))

    oof = pd.concat(oof, ignore_index=True)[["fold", "ID", "customer_id", "seen", "y", "p"]]
    threshold, _, _ = best_threshold(oof["y"], oof["p"])
    table = pd.DataFrame(rows).set_index("fold")
    table["f1"] = [f1_score(part["y"], part["p"] >= threshold) for _, part in oof.groupby("fold")]
    return table, oof, threshold

def pick_cache(cache, params):
    """A search can switch between caches (e.g. all or selected columns) through a function of its settings."""
    return cache(params) if callable(cache) else cache

def tune(cache, make_model, suggest, objective, name, n_trials=N_TRIALS):
    """Run or resume a stored Optuna search steered by log-loss or F1; every trial records both."""
    if FRESH_SEARCH:
        try:
            optuna.delete_study(study_name=name, storage=STORAGE)
        except KeyError:
            pass

    study = optuna.create_study(
        study_name=name, storage=STORAGE, load_if_exists=True, direction=OBJECTIVES[objective],
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),  # same seed, same trials on a rerun
    )
    expected = {**STUDY_METADATA, "objective": objective}
    if study.trials and any(study.user_attrs.get(k) != v for k, v in expected.items()):
        raise ValueError(f"{name}: stored search was run with other settings. Use a new experiment version.")
    for key, value in expected.items():
        study.set_user_attr(key, value)

    def run_trial(trial):
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)  # a model that does not converge stops the search
            params = suggest(trial)
            table, oof, threshold = evaluate_cached(pick_cache(cache, params), make_model(params))
        metrics = summary_row(table, oof, threshold)
        for key in ["f1", "log_loss", "roc_auc", "balanced_accuracy", "threshold", "overfit_gap"]:
            trial.set_user_attr(key, float(metrics[key]))
        return metrics[objective]

    done = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
    if done < n_trials:
        study.optimize(run_trial, n_trials=n_trials - done, show_progress_bar=False)
    print(f"{name}: {len(study.trials)} trials")
    return study


def trials_table(study):
    """Completed trials, best first by the steering objective."""
    done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    table = pd.DataFrame([{"trial": t.number, **t.params, **t.user_attrs} for t in done]).set_index("trial")
    objective = study.user_attrs["objective"]
    return table.sort_values(objective, ascending=OBJECTIVES[objective] == "minimize", kind="stable")


def shortlist(studies, cache, make_model, k=5):
    """Rescore the k best trials of each search, named like 'f1_trial_12'."""
    runs = {}
    for objective, study in studies.items():
        # Trials with identical scores are the same model in practice; keep one of each
        distinct = trials_table(study).round({"f1": 4, "log_loss": 4}).drop_duplicates(subset=["f1", "log_loss"])
        for number in distinct.head(k).index:
            params = study.trials[number].params
            runs[f"{objective}_trial_{number}"] = (evaluate_cached(pick_cache(cache, params), make_model(params)), params)
    return runs


def choose_winner(runs, reference):
    """Keep the reference unless a candidate is better; among better ones, fewer columns then higher F1."""
    board = judge(runs, reference)
    better = board.index[board["verdict"] == "better"].tolist()
    if not better:
        return reference, board
    best = board.loc[better].sort_values(["n_features", "f1"], ascending=[True, False]).index[0]
    return best, board

## 4.1. Naive Bayes

The default model's main problem was overconfidence. Two settings address it: `var_smoothing`, which adds a share of the largest variance to every variance, and calibration, which relearns the probabilities on held-out parts of the training years (sigmoid or isotonic). The elastic-net columns are kept.

In [24]:
# decision: Naive Bayes keeps the elastic-net selection (better than all columns at defaults)
nb_cache = FoldCache(Pipeline(build_linear_pipeline(GaussianNB()).steps[:-1]), linear_inputs)


def suggest_nb(trial):
    return {
        "var_smoothing": trial.suggest_float("var_smoothing", 1e-12, 1.0, log=True),
        "calibration": trial.suggest_categorical("calibration", ["none", "sigmoid", "isotonic"]),
    }


def make_nb(params):
    model = GaussianNB(var_smoothing=params["var_smoothing"])
    if params["calibration"] == "none":
        return model
    # Calibration learns on 5 internal splits of the training years only
    return CalibratedClassifierCV(model, method=params["calibration"], cv=5)


study_nb = {objective: tune(nb_cache, make_nb, suggest_nb, objective, f"naive_bayes_{objective}")
            for objective in OBJECTIVES}

for objective, study in study_nb.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5)[["var_smoothing", "calibration", "f1", "log_loss", "roc_auc", "threshold"]]
    display(top.style.format({"var_smoothing": "{:.1e}"}, precision=3))

naive_bayes_log_loss: 60 trials
naive_bayes_f1: 60 trials

Steered by log_loss


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
54,1.0e-12,isotonic,0.725,0.593,0.756,0.310
41,1.0e-12,isotonic,0.725,0.593,0.756,0.310
51,1.1e-12,isotonic,0.725,0.593,0.756,0.310
42,1.1e-12,isotonic,0.725,0.593,0.756,0.310
52,1.1e-12,isotonic,0.725,0.593,0.756,0.310



Steered by f1


,var_smoothing,calibration,f1,log_loss,roc_auc,threshold
trial,,,,,,
53,3.2e-05,isotonic,0.727,0.593,0.756,0.320
54,6.1e-05,isotonic,0.727,0.593,0.756,0.320
55,5.2e-05,isotonic,0.727,0.593,0.756,0.320
1,7.5e-11,isotonic,0.725,0.593,0.756,0.310
2,1.6e-05,isotonic,0.725,0.593,0.756,0.310


Both searches settle on isotonic calibration with negligible smoothing (below 10⁻⁴). The best trials reach F1 0.725–0.727 and log-loss 0.593, against 0.703 and 2.568 for the default. Of the 60 trials of each search, 37–39 tried isotonic calibration and none of the uncalibrated ones passed F1 0.708.

In [25]:
nb_candidates = shortlist(study_nb, nb_cache, make_nb)
runs = {"default": nb_runs["naive_bayes_selected"], **{name: run for name, (run, _) in nb_candidates.items()}}
chosen_nb, board_nb_tuned = choose_winner(runs, reference="default")
nb_params = nb_candidates[chosen_nb][1] if chosen_nb != "default" else {}

display(board_nb_tuned[["f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_nb} {nb_params}")

,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,
default,0.703,0.000,NaN,NaN,0,2.568,0.756,reference
log_loss_trial_54,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference
log_loss_trial_7,0.711,0.008,-0.001,0.023,1,0.595,0.759,no clear difference
log_loss_trial_16,0.713,0.010,-0.023,0.064,2,0.604,0.746,no clear difference
log_loss_trial_24,0.720,0.018,-0.011,0.066,1,0.611,0.747,no clear difference
log_loss_trial_40,0.723,0.020,-0.009,0.069,1,0.616,0.756,no clear difference
f1_trial_53,0.727,0.024,-0.004,0.071,1,0.593,0.756,no clear difference
f1_trial_1,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference
f1_trial_36,0.725,0.022,-0.007,0.071,1,0.593,0.756,no clear difference


Chosen: default {}


Calibration raises F1 by about +0.024, but the customer-bootstrap interval [-0.004, 0.071] includes zero, so under the decision rule the default stays. ROC-AUC does not move (0.756), because isotonic calibration keeps the order of the predictions and only changes their scale.

In [26]:
# Reliability on the pooled development predictions: default against the best calibrated trial
best_calibrated = board_nb_tuned.drop(index="default").sort_values("f1", ascending=False).index[0]
curves = {"default (uncalibrated)": runs["default"][1], f"calibrated ({best_calibrated})": runs[best_calibrated][1]}

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration",
                         line=dict(dash="dash", color="grey")))
brier = {}
for name, oof in curves.items():
    observed, predicted = calibration_curve(oof["y"], oof["p"], n_bins=10, strategy="quantile")
    brier[name] = brier_score_loss(oof["y"], oof["p"])
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode="lines+markers", name=name,
                             hovertemplate="predicted %{x:.2f}<br>observed %{y:.2f}<extra>" + name + "</extra>"))
fig.update_layout(
    title="Naive Bayes: predicted against observed repeat share (deciles of the predictions)",
    xaxis=dict(title="Mean predicted probability", range=[0, 1]),
    yaxis=dict(title="Observed repeat share", range=[0, 1]),
    height=480,
)
fig.write_html(FIGURES_DIR / "04_naive_bayes_calibration.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_naive_bayes_calibration.png", **PNG_OPTIONS)
fig.show()

print({name: round(value, 3) for name, value in brier.items()})

{'default (uncalibrated)': 0.287, 'calibrated (f1_trial_53)': 0.203}


The uncalibrated model pushes almost every prediction to 0 or 1: its lower deciles all predict about 0.00 while 29–43% of those purchases are followed by a repeat. After calibration the points follow the diagonal, from about 0.19 predicted against 0.31 observed in the lowest decile to 0.96 against 0.95 in the highest. The Brier score falls from 0.287 to 0.203.

## 4.2. Linear discriminant analysis

The default LDA ran on a singular covariance matrix. Shrinkage pulls the estimated covariance towards a diagonal one, which makes it invertible and more stable. The search tries no shrinkage, the automatic Ledoit–Wolf amount and fixed amounts from 0.0001 to 1, on all prepared columns or on the elastic-net selection.

In [31]:
# Two cached input sets: all prepared columns, or the elastic-net selection
lda_caches = {
    "all": FoldCache(build_linear_pipeline(LinearDiscriminantAnalysis(), select=False).named_steps["prep"], linear_inputs),
    "selected": FoldCache(Pipeline(build_linear_pipeline(LinearDiscriminantAnalysis()).steps[:-1]), linear_inputs),
}


def suggest_lda(trial):
    params = {
        "columns": trial.suggest_categorical("columns", ["all", "selected"]),
        "shrinkage_mode": trial.suggest_categorical("shrinkage_mode", ["none", "auto", "value"]),
    }
    if params["shrinkage_mode"] == "value":
        params["shrinkage"] = trial.suggest_float("shrinkage", 1e-4, 1.0, log=True)
    return params


def make_lda(params):
    if params["shrinkage_mode"] == "none":
        return LinearDiscriminantAnalysis(solver="svd")
    # Shrinkage needs a solver that estimates the covariance: lsqr
    shrinkage = "auto" if params["shrinkage_mode"] == "auto" else params["shrinkage"]
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage=shrinkage)


get_lda_cache = lambda params: lda_caches[params["columns"]]

study_lda = {objective: tune(get_lda_cache, make_lda, suggest_lda, objective, f"lda_{objective}")
             for objective in OBJECTIVES}

for objective, study in study_lda.items():
    print(f"\nSteered by {objective}")
    top = trials_table(study).head(5).reindex(
        columns=["columns", "shrinkage_mode", "shrinkage", "f1", "log_loss", "roc_auc", "threshold"]
    )
    display(top.style.format({"shrinkage": "{:.3g}"}, precision=3, na_rep="-"))

lda_log_loss: 60 trials
lda_f1: 60 trials

Steered by log_loss


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
54,selected,value,0.113,0.735,0.586,0.759,0.310
55,selected,value,0.195,0.735,0.587,0.760,0.310
58,selected,value,0.202,0.735,0.587,0.760,0.310
59,selected,value,0.204,0.735,0.587,0.760,0.310
56,selected,value,0.218,0.736,0.587,0.760,0.310



Steered by f1


,columns,shrinkage_mode,shrinkage,f1,log_loss,roc_auc,threshold
trial,,,,,,,
32,all,value,0.0211,0.740,0.594,0.756,0.350
33,all,value,0.021,0.740,0.594,0.756,0.350
36,all,value,0.021,0.740,0.594,0.756,0.350
50,all,value,0.0193,0.740,0.594,0.756,0.350
51,all,value,0.0187,0.739,0.594,0.756,0.350


The two searches find different optima. Log-loss prefers the selected columns with moderate shrinkage (about 0.1–0.2, log-loss 0.586–0.587). F1 prefers all columns with light shrinkage (about 0.02, F1 0.740). Every top trial uses a fixed shrinkage amount rather than none or the automatic one.

In [32]:
lda_candidates = shortlist(study_lda, get_lda_cache, make_lda)
runs = {"default": lda_runs["lda_selected"], **{name: run for name, (run, _) in lda_candidates.items()}}
chosen_lda, board_lda_tuned = choose_winner(runs, reference="default")
lda_params = lda_candidates[chosen_lda][1] if chosen_lda != "default" else {}

display(board_lda_tuned[["n_features", "f1", "f1_diff", "ci_low", "ci_high", "folds_worse", "log_loss", "roc_auc", "verdict"]])
print(f"Chosen: {chosen_lda} {lda_params}")

,n_features,f1,f1_diff,ci_low,ci_high,folds_worse,log_loss,roc_auc,verdict
model,,,,,,,,,
default,20.000,0.733,0.000,NaN,NaN,0,0.593,0.752,reference
log_loss_trial_54,20.000,0.735,0.002,-0.006,0.009,3,0.586,0.759,no clear difference
log_loss_trial_55,20.000,0.735,0.002,-0.007,0.009,2,0.587,0.760,no clear difference
log_loss_trial_58,20.000,0.735,0.002,-0.008,0.009,2,0.587,0.760,no clear difference
log_loss_trial_59,20.000,0.735,0.002,-0.008,0.009,2,0.587,0.760,no clear difference
log_loss_trial_56,20.000,0.736,0.003,-0.006,0.011,2,0.587,0.760,no clear difference
f1_trial_32,69.250,0.740,0.007,-0.006,0.016,1,0.594,0.756,no clear difference
f1_trial_50,69.250,0.740,0.007,-0.006,0.016,1,0.594,0.756,no clear difference
f1_trial_51,69.250,0.739,0.006,-0.006,0.016,1,0.594,0.756,no clear difference


Chosen: default {}


No candidate passes the rule. The best gain in F1 is +0.007 (0.740 against 0.733), with an interval of [-0.006, 0.016] and worse in one year. The log-loss candidates gain +0.002 to +0.003 and are worse in two or three years. The default LDA on the selected columns stays.

In [33]:
# F1 and log-loss across the shrinkage values tried (both searches, fixed-value trials only)
tried = pd.concat([trials_table(study) for study in study_lda.values()])
tried = tried.loc[tried["shrinkage_mode"] == "value"].sort_values("shrinkage")

fig = make_subplots(rows=1, cols=2, subplot_titles=["Pooled F1", "Mean validation log-loss"])
for i, (columns, part) in enumerate(tried.groupby("columns")):
    for col, metric in enumerate(["f1", "log_loss"], start=1):
        fig.add_trace(go.Scatter(
            x=part["shrinkage"], y=part[metric], mode="markers", name=f"{columns} columns",
            marker=dict(color=SERIES_COLOURS[i], size=8), legendgroup=columns, showlegend=col == 1,
            hovertemplate="shrinkage %{x:.3g}<br>" + metric + " %{y:.3f}<extra>" + columns + "</extra>",
        ), row=1, col=col)
fig.update_xaxes(type="log", title="Shrinkage (log scale)")
fig.update_layout(title="LDA: effect of shrinkage on F1 and log-loss", height=430)
fig.write_html(FIGURES_DIR / "04_lda_shrinkage.html", **HTML_OPTIONS)
fig.write_image(FIGURES_DIR / "04_lda_shrinkage.png", **PNG_OPTIONS)
fig.show()

F1 barely moves with shrinkage: 0.723–0.740 on all columns and 0.732–0.736 on the selected ones. Log-loss shows the trade-off more clearly. On all columns it falls from about 0.61 with almost no shrinkage to 0.589 around 0.1, then rises sharply towards 1.51 at full shrinkage, where the covariance becomes diagonal and the model ignores how the columns move together. The selected columns are less sensitive (0.586–0.625).